# COCO Detection DataLoader Demo
This notebook demonstrates the refactored `COCODetectionDataset` and the unified `Batch` dataclass.

In [ ]:
import torch
from omegaconf import OmegaConf
from yolo.config.config import DataConfig, DatasetConfig
from yolo.data.loader import create_dataloader
from yolo.utils.drawer import draw_bboxes
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
# Load configurations
coco_yaml_path = "/home/shrey/projects/yolo/yolo/config/dataset/coco.yaml"
dataset_cfg_dict = OmegaConf.to_container(OmegaConf.load(coco_yaml_path), resolve=True)
dataset_cfg = DatasetConfig(**dataset_cfg_dict)

data_cfg = DataConfig(
    shuffle=True,
    batch_size=4,
    pin_memory=False,
    dataloader_workers=0,
    image_size=[640, 640],
    data_augment={},
    source=None,
    dynamic_shape=False
)

print(f"✅ Loading dataset from: {dataset_cfg.path}")

In [ ]:
dataloader = create_dataloader(data_cfg, dataset_cfg, task="detect")
batch = next(iter(dataloader))

print(f"Batch object: {type(batch)}")
print(f"Images shape: {batch.images.shape}")
print(f"Targets shape: {batch.targets.shape}")

In [ ]:
num_to_show = min(2, batch.batch_size)
fig, axes = plt.subplots(1, num_to_show, figsize=(16, 8))
if num_to_show == 1: axes = [axes]

for i in range(num_to_show):
    img_tensor = batch.images[i]
    targets = batch.targets[i]
    
    # Filter out empty targets (class -1)
    valid_targets = targets[targets[:, 0] != -1]
    
    img_with_boxes = draw_bboxes(
        img_tensor, 
        valid_targets.tolist(), 
        idx2label=dataset_cfg.class_list
    )
    
    axes[i].imshow(img_with_boxes)
    axes[i].set_title(f"File: {Path(batch.paths[i]).name}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()